# Sales Analysis

Deployed by DCM as a Code Bundle. Run with:

```sql
EXECUTE CODE BUNDLE SALES_CICD_DEV.APPS.SALES_ANALYSIS
  ENTRYPOINT = 'sales_analysis.ipynb' ARGUMENTS = ('--env', 'DEV');
```

In [ ]:
import sys
from snowflake.snowpark.context import get_active_session

# Environment comes from EXECUTE CODE BUNDLE ... ARGUMENTS = ('--env', 'DEV'|'PROD')
args = sys.argv[1:]
env = args[args.index("--env") + 1].upper() if "--env" in args else "DEV"
if env not in ("DEV", "PROD"):
    raise ValueError(f"Unsupported env: {env}")

db = f"SALES_CICD_{env}"
session = get_active_session()
session.sql(f"USE WAREHOUSE SALES_CICD_WH_{env}").collect()
session.sql(f"USE SCHEMA {db}.ANALYTICS").collect()
print(f"Running sales analysis against {db}")

In [ ]:
summary = session.sql(f"""
    SELECT *
    FROM SEMANTIC_VIEW({db}.ANALYTICS.SV_SALES
        METRICS orders.TOTAL_REVENUE, orders.ORDER_COUNT
        DIMENSIONS orders.ORDER_DATE)
    ORDER BY ORDER_DATE DESC
    LIMIT 14
""").to_pandas()
print(summary.to_string(index=False))